# Fitbit daily averages
This notebook fetches heart rate (intraday when available), steps, and sleep from the Fitbit API for a date range, computes daily average heart rate, and saves CSV files. Set ACCESS_TOKEN and the date range, then run the code cell.

In [1]:
# Try to set start_date_str to the Fitbit account creation/memberSince date (if available),
# and set end_date_str to today. If found, overwrite the existing start_date_str/end_date_str.
import pandas as pd
import requests
from datetime import datetime, timedelta
import time
import os
import time 
import json
import math
import requests
ACCESS_TOKEN = os.environ.get("FITBIT_API")  # <-- replace with your token



In [ ]:
# Fetch functions: range-based (steps+heart) and rate-limited per-day sleep

# Token via env var or replace placeholder (not recommended)
HEADERS = {'Authorization': f'Bearer {ACCESS_TOKEN}', 'Accept': 'application/json'}
RANGE_DIR = 'fitbit_range_chunks'   # for steps and heart range responses
SLEEP_DIR = 'fitbit_sleep_chunks'  # rate-limited per-day sleep saved in 90-day grouped files
os.makedirs(RANGE_DIR, exist_ok=True)
os.makedirs(SLEEP_DIR, exist_ok=True)

# Configuration
START_DATE = datetime.strptime('2024-12-01', '%Y-%m-%d').date()
END_DATE = datetime.strptime('2026-04-30', '%Y-%m-%d').date()
RANGE_CHUNK_DAYS = 90  # use 90-day range calls for steps and heart
SLEEP_CHUNK_DAYS = 90  # group sleep per 90-day files but fetch per-day
USER_LIMIT_PER_HOUR = 150  # requests/hour for sleep enforcement
SLEEP_INTERVAL = math.ceil(3600 / USER_LIMIT_PER_HOUR)  # seconds between sleep requests

def fetch_range_chunks(path_fn, start, end, chunk_days, out_dir, prefix, pause=1.2):
    cur = start
    while cur <= end:
        chunk_end = min(end, cur + timedelta(days=chunk_days-1))
        s = cur.isoformat()
        e = chunk_end.isoformat()
        print(f'Fetching {prefix} range {s} to {e}')
        try:
            url = path_fn(s, e)
            r = requests.get(url, headers=HEADERS, timeout=90)
            r.raise_for_status()
            data = r.json()
            fname = f'{prefix}_{s}_to_{e}.json'
            with open(os.path.join(out_dir, fname), 'w', encoding='utf-8') as fh:
                json.dump(data, fh, ensure_ascii=False, indent=2)
        except Exception as exc:
            print(f'  Error fetching {prefix} {s} to {e}: {exc}')
        cur = chunk_end + timedelta(days=1)
        time.sleep(pause)

def fetch_sleep_rate_limited(start, end, chunk_days, out_dir, interval_seconds):
    import math
    cur = start
    while cur <= end:
        chunk_end = min(end, cur + timedelta(days=chunk_days-1))
        s = cur.isoformat()
        e = chunk_end.isoformat()
        print(f'Fetching sleep {s} to {e} (rate-limited, ~{interval_seconds}s/request)')
        items = []
        for n in range((chunk_end - cur).days + 1):
            day = cur + timedelta(days=n)
            day_s = day.isoformat()
            url = f'https://api.fitbit.com/1.2/user/-/sleep/date/{day_s}.json'
            attempts = 0
            while True:
                try:
                    r = requests.get(url, headers=HEADERS, timeout=60)
                    if r.status_code == 429:
                        # rate limit hit: exponential backoff
                        backoff = min(600, (2 ** attempts) * interval_seconds)
                        print(f'  429 received for {day_s}, sleeping {backoff}s before retry')
                        time.sleep(backoff)
                        attempts += 1
                        continue
                    r.raise_for_status()
                    items.append({'date': day_s, 'data': r.json()})
                    break
                except requests.exceptions.RequestException as exc:
                    attempts += 1
                    if attempts > 5:
                        print(f'  Failed to fetch {day_s} after {attempts} attempts: {exc}')
                        return
                        #break
                    wait = min(600, 2 ** attempts)
                    print(f'  Request error for {day_s}: {exc}; retrying in {wait}s')
                    time.sleep(wait)
            time.sleep(interval_seconds)
        out_file = f'sleep_{s}_to_{e}.json'
        with open(os.path.join(out_dir, out_file), 'w', encoding='utf-8') as fh:
            json.dump({'start': s, 'end': e, 'items': items}, fh, ensure_ascii=False, indent=2)
        cur = chunk_end + timedelta(days=1)
        time.sleep(1.0)



In [3]:
# Run: 1) fetch heart and steps by 90-day range chunks (no per-day rate enforcement)
fetch_range_chunks(lambda s,e: f'https://api.fitbit.com/1/user/-/activities/heart/date/{s}/{e}.json', START_DATE, END_DATE, RANGE_CHUNK_DAYS, RANGE_DIR, 'heart')
fetch_range_chunks(lambda s,e: f'https://api.fitbit.com/1/user/-/activities/steps/date/{s}/{e}.json', START_DATE, END_DATE, RANGE_CHUNK_DAYS, RANGE_DIR, 'steps')

Fetching heart range 2024-12-01 to 2025-02-28
Fetching heart range 2025-03-01 to 2025-05-29
Fetching heart range 2025-05-30 to 2025-08-27
Fetching heart range 2025-08-28 to 2025-11-25
Fetching heart range 2025-11-26 to 2026-02-23
Fetching heart range 2026-02-24 to 2026-04-30
Fetching steps range 2024-12-01 to 2025-02-28
Fetching steps range 2025-03-01 to 2025-05-29
Fetching steps range 2025-05-30 to 2025-08-27
Fetching steps range 2025-08-28 to 2025-11-25
Fetching steps range 2025-11-26 to 2026-02-23
Fetching steps range 2026-02-24 to 2026-04-30


In [ ]:
# Run: 2) fetch sleep per-day but grouped into 90-day files, enforcing the USER_LIMIT_PER_HOUR rate
#time.sleep(1200) #delete this line of code afterwards

fetch_sleep_rate_limited(START_DATE, END_DATE, SLEEP_CHUNK_DAYS, SLEEP_DIR, SLEEP_INTERVAL)

Fetching sleep 2024-12-01 to 2025-02-28 (rate-limited, ~24s/request)
Fetching sleep 2025-03-01 to 2025-05-29 (rate-limited, ~24s/request)
Fetching sleep 2025-05-30 to 2025-08-27 (rate-limited, ~24s/request)
Fetching sleep 2025-08-28 to 2025-11-25 (rate-limited, ~24s/request)
Fetching sleep 2025-11-26 to 2026-02-23 (rate-limited, ~24s/request)
Fetching sleep 2026-02-24 to 2026-04-30 (rate-limited, ~24s/request)


In [5]:
# Merge range-based steps+heart and rate-limited sleep into separate datasets and a combined CSV
import os, json
import pandas as pd
from datetime import datetime

# Directories used by the fetch cell
RANGE_DIR = './data/fitbit_range_chunks'   # contains heart_*.json and steps_*.json (range responses)
SLEEP_DIR = './data/fitbit_sleep_chunks'   # contains sleep_YYYY-MM-DD grouped chunk files with per-day items

steps_by_date = {}
hr_by_date = {}
sleep_rows = []  # list of {'date':..., 'sleep_minutes':..., ...} for sleep dataset

def load_json(path):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f'Could not load {path}: {e}')
        return None

# 1) Parse range files for steps and heart
if os.path.isdir(RANGE_DIR):
    for fn in sorted(os.listdir(RANGE_DIR)):
        path = os.path.join(RANGE_DIR, fn)
        j = load_json(path)
        if not j:
            continue
        lower = fn.lower()
        if 'steps' in lower:
            # activities-steps range response: list under 'activities-steps' or the response itself
            entries = j.get('activities-steps') or j.get('activities-steps') or []
            # sometimes range returns a list of dicts under 'activities-steps'
            if isinstance(entries, list):
                for ent in entries:
                    dt = ent.get('dateTime') or ent.get('date')
                    try:
                        steps_by_date[dt] = int(ent.get('value', 0)) if dt else None
                    except Exception:
                        steps_by_date[dt] = None
        elif 'heart' in lower:
            # activities-heart range response: entries under 'activities-heart'
            entries = j.get('activities-heart') or []
            for ent in entries:
                dt = ent.get('dateTime') or ent.get('date')
                avg = None
                v = ent.get('value') or {}
                if isinstance(v, dict):
                    avg = v.get('restingHeartRate')
                elif isinstance(v, (int, float)):
                    avg = v
                if avg is not None and dt:
                    try:
                        hr_by_date[dt] = float(avg)
                    except Exception:
                        hr_by_date[dt] = None



# 3) Build DataFrames and save separate CSVs
df_steps_heart = pd.DataFrame([{'date': k, 'steps': v, 'avg_heart_rate': hr_by_date.get(k)} for k,v in steps_by_date.items()])
if df_steps_heart.empty:
    print('No steps/heart range data found in', RANGE_DIR)
else:
    df_steps_heart['date'] = pd.to_datetime(df_steps_heart['date'])
    df_steps_heart = df_steps_heart.sort_values('date').reset_index(drop=True)
    df_steps_heart.to_csv('fitbit_steps_heart.csv', index=False)
    print('Wrote fitbit_steps_heart.csv with', len(df_steps_heart), 'rows')



Wrote fitbit_steps_heart.csv with 516 rows


In [16]:
# 2) Parse sleep chunk files (each contains 'items' list with per-day 'data')
sleep_rows = []
df_sleep = pd.DataFrame()
if os.path.isdir(SLEEP_DIR):
    for fn in sorted(os.listdir(SLEEP_DIR)):
        path = os.path.join(SLEEP_DIR, fn)
        j = load_json(path)
        if not j:
            continue
        items = j.get('items') or []
        for it in items:
            d = it.get('date')
            data = it.get('data') or {}
            # sleep responses often include 'summary' and/or 'sleep' list
            mins = None
            summary = data.get('summary', {})
            #print(summary)
            if summary:
                mins = summary.get('totalMinutesAsleep') or summary.get('totalTimeInBed')
            if mins is None and 'sleep' in data and isinstance(data.get('sleep'), list) and len(data['sleep'])>0:
                rec = data['sleep'][0]
                mins = rec.get('minutesAsleep') or rec.get('totalMinutesAsleep')
            if mins is not None and d:
                try:
                    sleep_rows.append({'date': d, 'sleep_minutes': int(mins)})
                except Exception:
                    sleep_rows.append({'date': d, 'sleep_minutes': None})
                    df_sleep = pd.DataFrame(sleep_rows)
                               
df_sleep = pd.DataFrame(sleep_rows)        
if df_sleep.empty:
    print('No sleep data found in', SLEEP_DIR)
else:
    df_sleep['date'] = pd.to_datetime(df_sleep['date'])
    df_sleep = df_sleep.sort_values('date').reset_index(drop=True)
    df_sleep.to_csv('fitbit_sleep.csv', index=False)
    print('Wrote fitbit_sleep.csv with', len(df_sleep), 'rows')



Wrote fitbit_sleep.csv with 516 rows


In [17]:
# 4) Merge on date into combined CSV (keep sleep and steps+heart separate before merge)
if not df_steps_heart.empty and not df_sleep.empty:
    df_merged = pd.merge(df_steps_heart, df_sleep[['date','sleep_minutes']], on='date', how='outer')
    df_merged = df_merged.sort_values('date').reset_index(drop=True)
    df_merged.to_csv('fitbit_merged.csv', index=False)
    print('Wrote fitbit_merged.csv with', len(df_merged), 'rows')
else:
    print('Merged CSV not created — missing one of the datasets.')

Wrote fitbit_merged.csv with 516 rows
